# 30 · Риск-портфель RBAM и план ТО (Фаза A MVP_PIVOT.md)

Ядро MVP — «где риск и что делать в план ТО», а не «что сломается через 24ч».

1. severity по типу канала (SEVERITY_MAP.md) + масштаб объекта;
2. score = risk30(cal) × severity × scale → топ-риски;
3. план ТО по кварталам (exp_days) и типу;
4. KPI-сет ядра (Precision@топ-K, Уно-С, калибровка, медианы);
5. генерация `research/docs/RISK_PIVOT_REPORT.md` для презентации заказчику.

In [1]:
%matplotlib inline
import json
import pathlib
import sys

_HERE = pathlib.Path.cwd()
RESEARCH = _HERE.parent if _HERE.name == "notebooks" else _HERE
if str(RESEARCH) not in sys.path:
    sys.path.insert(0, str(RESEARCH))

import pandas as pd
import numpy as np
import inference_contract as ic
import data_utils as du

TASKS = ("fire", "access", "sensor", "wear")
MODELS = RESEARCH / "models"
DOCS = RESEARCH / "docs"


In [2]:
# 1. Топ-риски RBAM по каждой задаче (score = risk × severity × scale)
for t in TASKS:
    v = ic.rbam_view(t, k=100)
    cols = ["тип_датчика", "severity", "risk_used", "exp_days", "score", "plan"]
    print("=== %s: топ-5 ===" % t)
    print(v[cols].head(5).round(3).to_string(index=False))
    print(dict(v["plan"].value_counts()))
    print()


=== fire: топ-5 ===
тип_датчика  severity  risk_used  exp_days  score            plan
Датчик дыма      0.85        1.0     0.221   0.85 текущий квартал
Датчик дыма      0.85        1.0     0.208   0.85 текущий квартал
Датчик дыма      0.85        1.0     0.217   0.85 текущий квартал
Датчик дыма      0.85        1.0     0.217   0.85 текущий квартал
Датчик дыма      0.85        1.0     0.217   0.85 текущий квартал
{'текущий квартал': np.int64(61), 'следующий квартал': np.int64(39)}

=== access: топ-5 ===
     тип_датчика  severity  risk_used  exp_days  score              plan
          КД Люк       0.7       0.88    11.874  0.616 следующий квартал
9-секционный люк       0.7       0.88     5.607  0.616   текущий квартал
9-секционный люк       0.7       0.88     7.760  0.616 следующий квартал
9-секционный люк       0.7       0.88    10.770  0.616 следующий квартал
9-секционный люк       0.7       0.88     7.712  0.616 следующий квартал
{'следующий квартал': np.int64(66), 'текущий квартал':

In [3]:
# 2. План ТО по кварталам (maintenance_plan) — пример JSON-контракта
mp = ic.maintenance_plan("wear", k=1000)
print("wear /maintenance-plan: rows=", len(mp["rows"]))
print(chr(10).join(str(r) for r in mp["rows"][:6]))

# проверка JSON-сериализации для всех задач (Фаза B контракт)
for t in TASKS:
    d = ic.top_risks(t, k=20)
    json.dumps(d, ensure_ascii=False)
    print("top_risks(%s) JSON OK, items=%d" % (t, len(d["items"])))


wear /maintenance-plan: rows= 2
{'plan': 'следующий квартал', 'тип_датчика': 'Состояние насоса', 'каналов': 976, 'риск_средний': 0.8961, 'score_сумма': 699.6593, 'exp_days_медиана': 10.7124}
{'plan': 'текущий квартал', 'тип_датчика': 'Состояние насоса', 'каналов': 24, 'риск_средний': 0.9738, 'score_сумма': 18.6962, 'exp_days_медиана': 6.3568}
top_risks(fire) JSON OK, items=20
top_risks(access) JSON OK, items=20
top_risks(sensor) JSON OK, items=20
top_risks(wear) JSON OK, items=20


In [4]:
# 3. KPI-сет ядра (из финальной таблицы) + флоу RBAM-плана в markdown
ft = ic.final_metrics()
lines = []
lines.append("# Риск-портфель RBAM — отчёт (20.09.2026, ядро MVP)")
lines.append("")
lines.append("Сервис отвечает: «где риск прямо сейчас и что делать в план ТО».")
lines.append("Горизонты: risk30 (P(≤30д)), exp_days (медиана дней), S(t).")
lines.append("")
lines.append("## KPI-сет (holdout ≥ 2025-07-01, 20k субъектов)")
lines.append("")
lines.append(ft.round(4).to_markdown(index=False))

In [5]:
# 4. Продолжение отчёта: топ-риски и планы ТО
for t in TASKS:
    v = ic.rbam_view(t, k=200)
    lines.append("")
    lines.append("## %s — топ-10 каналов по RBAM-score" % t)
    lines.append("")
    c = ["тип_датчика", "severity", "scale", "risk_used", "exp_days", "score", "plan", "дата"]
    lines.append(v[c].head(10).round(3).to_markdown(index=False))
    mp = ic.maintenance_plan(t, k=1000)
    lines.append("")
    lines.append("План ТО (агрегат plan × тип):")
    lines.append(pd.DataFrame(mp["rows"]).to_markdown(index=False))

lines.append("")
lines.append("Формула: score = risk30(cal или raw) × severity(тип) × scale(объект).")
lines.append("severity-маппинг: SEVERITY_MAP.md; реализация: inference_contract.rbam_view.")
out = DOCS / "RISK_PIVOT_REPORT.md"
out.write_text("\n".join(lines), encoding="utf-8")
print("отчёт сохранён:", out)


отчёт сохранён: D:\Downloads_D\lct_a8\research\docs\RISK_PIVOT_REPORT.md
